# Create FewRel Relation Split and Dev Episodes

This notebook reads the raw FewRel train JSON, splits relations into train/dev relation sets, writes relation-grouped intermediate JSON files, and samples full-object few-shot dev episodes.

It does not mirror the final pickle format directly; run `process_FewRel.ipynb` after this notebook to create the train and dev pickle files used by the existing pipeline.


In [1]:
from pathlib import Path
import json
import random
from collections import defaultdict, Counter
import numpy as np

RAW_FEWREL_PATH = Path('/storage2/home/aunabilchakma/data/relation_extraction/fewrel_train_none_entity.json')
DATA_DIR = Path('../data')

# FewRel train has 64 relations x 700 examples. Keep most relations for training and hold out the rest for dev episodes.
TRAIN_RELATION_COUNT = 54
DEV_RELATION_COUNT = 10
RELATION_SPLIT_SEED = 20260517

# Episode settings matching the existing fs_tacred_final_step_dev_* naming/shape.
NO_RELATION_LABEL = 'no_relation'
N_WAY = 5
K_SHOT = 1
NUM_QUERIES_PER_EPISODE = 1
NUM_DEV_EPISODES = 9000
EPISODE_SEED = 160292

RELATION_SPLIT_PATH = DATA_DIR / 'fewrel_relation_split.json'
TRAIN_RELATION_JSON_PATH = DATA_DIR / 'fewrel_train_relation_samples.json'
DEV_RELATION_JSON_PATH = DATA_DIR / 'fewrel_dev_relation_samples.json'
DEV_EPISODES_JSON_PATH = DATA_DIR / f'fewrel_dev_{N_WAY}_way_{K_SHOT}_shots_{NUM_DEV_EPISODES}episodes_seed{EPISODE_SEED}.json'


In [2]:
def read_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def save_json(data, path, indent=2):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=indent, ensure_ascii=False)

def has_valid_entity_spans(sample):
    tokens = sample.get('token', [])
    n_tokens = len(tokens)
    span_fields = ['subj_start', 'subj_end', 'obj_start', 'obj_end']
    if n_tokens == 0:
        return False
    if any(field not in sample for field in span_fields):
        return False
    if not all(isinstance(sample[field], int) for field in span_fields):
        return False
    if sample['subj_start'] > sample['subj_end'] or sample['obj_start'] > sample['obj_end']:
        return False
    return (
        0 <= sample['subj_start'] < n_tokens
        and 0 <= sample['subj_end'] < n_tokens
        and 0 <= sample['obj_start'] < n_tokens
        and 0 <= sample['obj_end'] < n_tokens
    )

def normalize_sample(sample):
    # Keep exactly the fields used by the TACRED pickle code. FewRel entity types are unavailable here, so they stay None.
    return {
        'id': sample['id'],
        'relation': sample['relation'],
        'token': list(sample['token']),
        'subj_start': sample['subj_start'],
        'subj_end': sample['subj_end'],
        'obj_start': sample['obj_start'],
        'obj_end': sample['obj_end'],
        'subj_type': sample.get('subj_type'),
        'obj_type': sample.get('obj_type'),
    }

def group_by_relation(samples):
    grouped = defaultdict(list)
    for sample in samples:
        item = normalize_sample(sample)
        grouped[item['relation']].append(item)
    return dict(grouped)


In [3]:
raw_samples_all = read_json(RAW_FEWREL_PATH)
raw_samples = [sample for sample in raw_samples_all if has_valid_entity_spans(sample)]
invalid_samples = [sample for sample in raw_samples_all if not has_valid_entity_spans(sample)]
by_relation = group_by_relation(raw_samples)
relation_counts = Counter({relation: len(samples) for relation, samples in by_relation.items()})

print('raw samples before span filter:', len(raw_samples_all))
print('raw samples after span filter:', len(raw_samples))
print('dropped invalid-span samples:', len(invalid_samples))
print('relations:', len(by_relation))
print('examples per relation:', sorted(set(relation_counts.values())))
print('duplicate ids:', len(raw_samples) - len({sample['id'] for sample in raw_samples}))
if invalid_samples:
    print('first invalid sample:', invalid_samples[0])

assert TRAIN_RELATION_COUNT + DEV_RELATION_COUNT == len(by_relation)
assert min(relation_counts.values()) >= K_SHOT + NUM_QUERIES_PER_EPISODE


raw samples before span filter: 44800
raw samples after span filter: 44780
dropped invalid-span samples: 20
relations: 64
examples per relation: [697, 698, 699, 700]
duplicate ids: 0
first invalid sample: {'id': 'P931_217', 'relation': 'place served by transport hub', 'subj_start': 3, 'subj_end': 4, 'obj_start': 20, 'obj_end': 21, 'token': ['Aviation', 'fuel', 'at', 'Williams', 'Field', 'is', 'pumped', 'in', 'a', '16', 'km', '(', '10', 'mi', ')', 'flexible', 'pipe', 'from', 'McMurdo', 'Station', '.'], 'subj_type': None, 'obj_type': None}


In [4]:
rng = random.Random(RELATION_SPLIT_SEED)
relations = sorted(by_relation)
rng.shuffle(relations)

train_relations = sorted(relations[:TRAIN_RELATION_COUNT])
dev_relations = sorted(relations[TRAIN_RELATION_COUNT:])

assert set(train_relations).isdisjoint(dev_relations)
assert len(train_relations) == TRAIN_RELATION_COUNT
assert len(dev_relations) == DEV_RELATION_COUNT

relation_split = {
    'raw_path': str(RAW_FEWREL_PATH),
    'relation_split_seed': RELATION_SPLIT_SEED,
    'train_relation_count': TRAIN_RELATION_COUNT,
    'dev_relation_count': DEV_RELATION_COUNT,
    'train_relations': train_relations,
    'dev_relations': dev_relations,
}

train_by_relation = {relation: by_relation[relation] for relation in train_relations}
dev_by_relation = {relation: by_relation[relation] for relation in dev_relations}

print('train examples:', sum(len(v) for v in train_by_relation.values()))
print('dev pool examples:', sum(len(v) for v in dev_by_relation.values()))
print('dev relations:', dev_relations)


train examples: 37780
dev pool examples: 7000
dev relations: ['after a work by', 'composer', 'distributor', 'language of work or name', 'licensed to broadcast to', 'location of formation', 'member of political party', 'occupant', 'subsidiary', 'winner']


In [5]:
def sample_dev_episodes(support_pool, train_pool, *, n_way, k_shot, num_queries, num_episodes, seed):
    # TACRED-style formula adapted for FewRel:
    # - sample N support ways uniformly from held-out dev relations
    # - sample query source labels from the N support ways plus an artificial no_relation pool
    # - build the artificial no_relation pool per episode from train relations plus dev relations outside the current ways
    random.seed(seed)
    np.random.seed(seed)

    support_relations = sorted(support_pool)
    support_weights = [1.0 / len(support_relations)] * len(support_relations)
    episodes = []

    for episode_idx in range(num_episodes):
        ways = np.random.choice(
            a=support_relations,
            size=n_way,
            replace=False,
            p=support_weights,
        ).tolist()

        meta_train = []
        support_by_relation = {}
        for relation in ways:
            support = random.sample(support_pool[relation], k_shot)
            meta_train.append(support)
            support_by_relation[relation] = {sample['id'] for sample in support}

        no_relation_pool = [
            sample
            for relation in sorted(train_pool)
            for sample in train_pool[relation]
        ]
        no_relation_pool.extend(
            sample
            for relation in support_relations
            if relation not in ways
            for sample in support_pool[relation]
        )

        query_source_labels = ways + [NO_RELATION_LABEL]
        query_counts = np.array([len(support_pool[relation]) for relation in ways] + [len(no_relation_pool)], dtype=float)
        query_weights = (query_counts / query_counts.sum()).tolist()
        query_source_relations = np.random.choice(
            a=query_source_labels,
            size=num_queries,
            replace=True,
            p=query_weights,
        ).tolist()

        meta_test = []
        query_episode_relations = []
        for source_relation in query_source_relations:
            if source_relation != NO_RELATION_LABEL:
                candidates = [
                    sample for sample in support_pool[source_relation]
                    if sample['id'] not in support_by_relation[source_relation]
                ]
                query = random.choice(candidates).copy()
                query_episode_relation = source_relation
            else:
                query = random.choice(no_relation_pool).copy()
                query['original_relation'] = query['relation']
                query['original_id'] = query['id']
                query['id'] = f"{query['id']}__no_relation"
                query['relation'] = NO_RELATION_LABEL
                query_episode_relation = NO_RELATION_LABEL

            meta_test.append(query)
            query_episode_relations.append(query_episode_relation)

        episodes.append({
            'episode_id': episode_idx,
            'ways': ways,
            'query_source_relations': query_source_relations,
            'query_episode_relations': query_episode_relations,
            'meta_train': meta_train,
            'meta_test': meta_test,
        })

    return episodes

dev_episodes = sample_dev_episodes(
    dev_by_relation,
    train_by_relation,
    n_way=N_WAY,
    k_shot=K_SHOT,
    num_queries=NUM_QUERIES_PER_EPISODE,
    num_episodes=NUM_DEV_EPISODES,
    seed=EPISODE_SEED,
)

no_relation_query_count = sum(
    query['relation'] == NO_RELATION_LABEL
    for episode in dev_episodes
    for query in episode['meta_test']
)
total_query_count = sum(len(episode['meta_test']) for episode in dev_episodes)
episodes_with_no_relation = sum(
    any(query['relation'] == NO_RELATION_LABEL for query in episode['meta_test'])
    for episode in dev_episodes
)

print('episodes:', len(dev_episodes))
print('first episode ways:', dev_episodes[0]['ways'])
print('first episode query source relations:', dev_episodes[0]['query_source_relations'])
print('first episode query episode relations:', dev_episodes[0]['query_episode_relations'])
print('no_relation queries:', no_relation_query_count, '/', total_query_count, f'({no_relation_query_count / total_query_count:.2%})')
print('episodes with no_relation:', episodes_with_no_relation, '/', len(dev_episodes), f'({episodes_with_no_relation / len(dev_episodes):.2%})')


episodes: 9000
first episode ways: ['composer', 'member of political party', 'subsidiary', 'language of work or name', 'winner']
first episode query source relations: ['no_relation']
first episode query episode relations: ['no_relation']
no_relation queries: 8277 / 9000 (91.97%)
episodes with no_relation: 8277 / 9000 (91.97%)


In [6]:
# Sanity checks before writing intermediates.
for episode in dev_episodes:
    support_ids = {sample['id'] for way in episode['meta_train'] for sample in way}
    query_ids = {sample['id'] for sample in episode['meta_test']}
    assert support_ids.isdisjoint(query_ids)
    assert len(episode['meta_train']) == N_WAY
    assert all(len(way) == K_SHOT for way in episode['meta_train'])
    assert len(episode['meta_test']) == NUM_QUERIES_PER_EPISODE
    assert all(sample['relation'] in episode['ways'] for way in episode['meta_train'] for sample in way)
    assert all(
        sample['relation'] == NO_RELATION_LABEL or sample['relation'] in episode['ways']
        for sample in episode['meta_test']
    )
    for source_relation, episode_relation, sample in zip(
        episode['query_source_relations'],
        episode['query_episode_relations'],
        episode['meta_test'],
    ):
        if source_relation in episode['ways']:
            assert episode_relation == source_relation
            assert sample['relation'] == source_relation
        else:
            assert source_relation == NO_RELATION_LABEL
            assert episode_relation == NO_RELATION_LABEL
            assert sample['relation'] == NO_RELATION_LABEL
            assert sample['id'].endswith('__no_relation')
            assert sample['original_id'] != sample['id']
            assert sample['original_relation'] not in episode['ways']
            assert sample['original_relation'] in train_by_relation or sample['original_relation'] in dev_by_relation

print('no_relation queries:', no_relation_query_count, '/', total_query_count, f'({no_relation_query_count / total_query_count:.2%})')

print('sanity checks passed')


no_relation queries: 8277 / 9000 (91.97%)
sanity checks passed


In [7]:
# Run this cell when you are ready to create the intermediate JSON files.
save_json(relation_split, RELATION_SPLIT_PATH)
save_json(train_by_relation, TRAIN_RELATION_JSON_PATH)
save_json(dev_by_relation, DEV_RELATION_JSON_PATH)
save_json(dev_episodes, DEV_EPISODES_JSON_PATH)

print('wrote:', RELATION_SPLIT_PATH)
print('wrote:', TRAIN_RELATION_JSON_PATH)
print('wrote:', DEV_RELATION_JSON_PATH)
print('wrote:', DEV_EPISODES_JSON_PATH)


wrote: ../data/fewrel_relation_split.json
wrote: ../data/fewrel_train_relation_samples.json
wrote: ../data/fewrel_dev_relation_samples.json
wrote: ../data/fewrel_dev_5_way_1_shots_9000episodes_seed160292.json
